<a href="https://colab.research.google.com/github/bhaviii123/Air_Passenger_Forecasting_ML_vs_DL.ipynb/blob/main/Copy_of_stock_market_prediction_using_time_series_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Nifty 50 data (Last 5 Years)
df = yf.download('^NSEI', start='2021-01-01', end='2026-03-01')
df = df[['Close']].dropna()

# Split into Train (80%) and Test (20%)
train_size = int(len(df) * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]

/tmp/ipython-input-182/3821275831.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('^NSEI', start='2021-01-01', end='2026-03-01')
[*********************100%***********************]  1 of 1 completed


Step 1: ARIMA (The Statistical Approach)

In [ ]:
# Install pmdarima if not already installed
!pip install pmdarima

from pmdarima import auto_arima

# 1. Fit Auto-ARIMA
arima_model = auto_arima(train, seasonal=False, stepwise=True)

# 2. Forecast
arima_pred_values = arima_model.predict(n_periods=len(test))
arima_pred = pd.Series(arima_pred_values, index=test.index)

# Output: A relatively straight line following the general direction of the train set.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


Step 2: Prophet (The Trend & Seasonality Approach)

In [ ]:
from prophet import Prophet
import pandas as pd

# 1. Prepare Data
# The error indicates that df_pr['y'] is not a Series (ndim > 1) when Prophet processes it.
# Explicitly create the DataFrame to ensure correct structure and 1-dimensional 'y' values.
df_pr = pd.DataFrame({'ds': train.index, 'y': train['Close'].values.flatten()})

# 2. Fit Model
m = Prophet(daily_seasonality=True)
m.fit(df_pr)

# 3. Create Future Dates and Predict
future = m.make_future_dataframe(periods=len(test))
forecast = m.predict(future)
prophet_pred = forecast.iloc[-len(test):]['yhat'].values

# Output: A forecast that captures weekly dips and monthly peaks.

Step 3: LSTM (The Deep Learning Approach)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# 1. Scale Data
scaler = MinMaxScaler()
scaled_train = scaler.fit_transform(train)

# 2. Create Sequences (60 days look-back)
X_train, y_train = [], []
for i in range(60, len(scaled_train)):
    X_train.append(scaled_train[i-60:i, 0])
    y_train.append(scaled_train[i, 0])
X_train, y_train = np.array(X_train), np.array(y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

# 3. Build & Train LSTM
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(60, 1)),
    Dropout(0.2),
    LSTM(50),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

# Output: A highly reactive line that mimics the volatility of the actual stock.

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
